## Powderday SED Workflow — PSB Galaxy Sample

End-to-end pipeline: filter particle files → write Slurm masters → run Powderday RT → extract multi-band photometric fluxes.

The **same** source list drives two runs that differ only in dust treatment:

- **dust_on**  — full dust model: SIMBA live dust masses (`dust_grid_type = 'manual'`, `n_photons_raytracing_dust = 1e7`)
- **dust_off** — dust blocked (`dust_grid_type = 'dtm'`, `dusttometals_ratio = 1e-12`, `1` photon for dust RT)

Neither run uses the Charlot & Fall birth-cloud dust (`CF_on = False` in both masters); the two parameter
masters are identical except the dust-grid type, the dust-to-metals ratio, and the dust-raytracing photon count.
Outputs go to a fresh `sed_dust_comparison/` folder so earlier runs are untouched.
Target sample: post-starburst galaxies from SIMBA-100.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from astropy.cosmology import LambdaCDM
import astropy.units as u
from astropy.io import fits
from astropy.table import Table
from scipy import interpolate

from simbanator.sed.makesed import MakeSED
from simbanator.io.simba import Simulation

cosmo = LambdaCDM(H0=68, Om0=0.3, Ode0=0.7, Ob0=0.048)
plt.rcParams.update({'font.size': 14})

sim = Simulation('cis100')

### Configuration — shared sample + the two dust runs

Both runs read the identical `(snap, galaxyID)` source list and write to **separate** `run_tag`
output trees, so they never overwrite each other. Each run only differs in which parameter master
it copies into the Powderday job dirs.

In [ ]:
GVFS_BASE   = ''#"/run/user/1000/gvfs/sftp:host=slurmusrint2.cis.gov.pl,user=glorenzon"
# Use '+' not os.path.join: with GVFS_BASE='' this must stay ABSOLUTE ('/mnt/...'), and
# os.path.join(GVFS_BASE, "mnt/...") yields a RELATIVE path that doubles against the CWD
# (e.g. .../analize_simba_cgm/mnt/home/glorenzon/...), so Stage 2 can't find the SEDs.
REMOTE_HOME = GVFS_BASE + "/mnt/home/glorenzon/analize_simba_cgm"

hydro_dir_base = os.path.join(os.getcwd(), 'output', sim.name, 'filtered_particles')
selection_file = 'selection_simba100_poststarburst.hdf5'
# fresh output folder so this dust-comparison experiment never overwrites previous runs (e.g. .../sed/PSBG_dust_off)
output_dir     = os.path.join(REMOTE_HOME, 'output', sim.name, 'sed_dust_comparison')
SAMPLE_FITS    = '/mnt/home/glorenzon/analize_simba_cgm/output/cis100/quenching_times/forward_modeled_unique_sample.fits'

# filtered-particle filename prefix — MUST be the same value passed to extract_particles (Stage 0)
# and create_master (Stage 1). Stage 0 saves <PREFIX>_snap<NNN>_gal<IDIDID>.h5; create_master bakes
# the identical name into each snap_*.py via the cluster template, so the two must never drift.
PARTICLE_PREFIX = 'm100n1024'

# the two runs: identical in every respect except the parameter master (dust treatment)
RUNS = {
    'dust_on':  dict(run_tag='dusty_simdust',  paramf='parameters_master.py'),
    'dust_off': dict(run_tag='nodust_1e-12',   paramf='parameters_master-nodust.py'),
}

# one MakeSED per run: same sim / hydro dir / selection file / output_dir, distinct run_tag
makeseds = {
    key: MakeSED(sim, nnodes=1, model_run_name=cfg['run_tag'],
                 hydro_dir_base=hydro_dir_base, selection_file=selection_file,
                 output_dir=output_dir, run_tag=cfg['run_tag'])
    for key, cfg in RUNS.items()
}
for key, ms in makeseds.items():
    print(f"{key:9s} -> run_tag='{ms.run_tag}', master='{RUNS[key]['paramf']}'")

### Shared source list (identical for both runs)

Read the PSB sample **once** and reuse the same `(SNAPS, IDS)` arrays for selection, master
generation, and flux extraction in both runs — this is what guarantees the two runs contain
exactly the same galaxies.

In [ ]:
with fits.open(SAMPLE_FITS) as f:
    _snaps = np.asarray(f[1].data['SNAPSHOT'], dtype=int)
    _ids   = np.asarray(f[1].data['GROUPID_SNAPSHOT'], dtype=int)

# sort + de-duplicate (snap, id) so the source list is canonical and identical for both runs
_pairs = np.unique(np.stack([_snaps, _ids], axis=1), axis=0)
SNAPS, IDS = _pairs[:, 0], _pairs[:, 1]

_ndup = len(_snaps) - len(SNAPS)
print(f"{len(SNAPS)} unique sources over {len(np.unique(SNAPS))} snapshots"
      + (f"  ({_ndup} duplicate pair(s) removed)" if _ndup else ""))

### Stage 0 — extract the shared particle files (once, used by both runs)

Powderday reads per-galaxy particle files from `hydro_dir_base/snap_NNN/`. They are **identical for
`dust_on` and `dust_off`** — the dust treatment is set by the parameter master, not the particle
data — so extract them **once** here, before Stage 1. `extract_particles` reads each snapshot a
single time, writes one HDF5 per galaxy (gas + stars; gas keeps its `Dust_Masses` field), and skips
files that already exist unless `EXTRACT_OVERWRITE=True`.

In [ ]:
from simbanator.analysis import extract_particles

# Stage 0 — extract the per-galaxy particle files ONCE. Powderday reads these from
# hydro_dir_base/snap_NNN/<PARTICLE_PREFIX>_snap<snap>_gal<id>.h5; they are IDENTICAL for dust_on and
# dust_off (the dust treatment lives in the parameter master, not the particles), so both runs share
# them. extract_particles writes under cwd/output/<sim>/filtered_particles == hydro_dir_base, one
# file per galaxy, reading each snapshot only once. Existing files are skipped unless overwrite=True.
#
# The prefix MUST match create_master's prefix (Stage 1): it is the value baked into each snap_*.py
# as snapshot_name. Both use PARTICLE_PREFIX so the saved name and the looked-up name can't diverge.
#
# Truncated/corrupt snapshot files (a half-staged download) raise OSError; we skip that snapshot,
# keep going, and list the casualties at the end so the run isn't lost to one bad file.
EXTRACT_OVERWRITE = False
EXTRACT_PTYPES    = ("PartType0", "PartType4")   # gas (carries Dust_Masses) + stars; add "PartType5" for AGN

bad_snaps = []                                    # (snap, n_gal, error) for unreadable snapshots
for _snap in np.unique(SNAPS):
    _snap = int(_snap)
    _ids_here = np.unique(IDS[SNAPS == _snap])
    _simfile = sim.get_snapshot_file(_snap)
    print(f"snap {_snap:3d}: extracting {len(_ids_here)} galaxies from {os.path.basename(_simfile)}")
    try:
        _cs = sim.load_catalog(snap=_snap)
        extract_particles(_cs, _simfile, _snap, galaxy_ids=_ids_here, ptypes=EXTRACT_PTYPES,
                          sim_name=sim.name, prefix=PARTICLE_PREFIX,
                          overwrite=EXTRACT_OVERWRITE, verbose=1)
        del _cs
    except (OSError, KeyError) as e:
        print(f"  [SKIP] snap {_snap}: {type(e).__name__}: {str(e).splitlines()[0]}")
        bad_snaps.append((_snap, len(_ids_here), f"{type(e).__name__}: {str(e).splitlines()[0]}"))

print("\nparticle extraction complete ->", hydro_dir_base)
if bad_snaps:
    n_gal_bad = sum(n for _, n, _ in bad_snaps)
    print(f"\n{len(bad_snaps)} snapshot(s) unreadable -> {n_gal_bad} galaxies skipped "
          f"(these (snap, id) pairs will be missing from BOTH runs, consistently):")
    for s, n, err in bad_snaps:
        ids_s = np.unique(IDS[SNAPS == s])
        print(f"  snap {s:3d}: {n} galaxies {list(ids_s)} | {err}")
    print("\nThese snapshot files are incomplete on disk (truncated). Re-stage/re-download them in")
    print("the shared SIMBA data dir, then re-run this cell (existing files are skipped).")

### Stage 1 — write the HDF5 selection and Slurm masters (both runs)

Run this, then launch the Powderday RT jobs on the cluster (`submit_all_snaps.sh` under each run's
`powderday_sed_out/`). Come back and run Stage 2 once the `.rtout.sed` files exist.

In [ ]:
for key, cfg in RUNS.items():
    ms = makeseds[key]
    print(f"\n=== {key} (run_tag='{ms.run_tag}', master='{cfg['paramf']}') ===")
    ms.selection_gals(snaps=SNAPS, galaxyID=IDS)                 # same sources for every run
    ms.create_master('cluster', 'plist', radius=None, partition='INTEL_SKYLAKE,INTEL_CASCADE,INTEL_PHI,INTEL_HASWELL',
                     prefix=PARTICLE_PREFIX, paramf=cfg['paramf'], snaps_to_run=None)  # same prefix Stage 0 saved with

### Filter set for flux extraction

`local_filters` supplements the built-in filter library; each entry is keyed as
`{telescope_label: {instrument_label: {filter_name: path_to_res_file}}}`.
Johnson V and U must use distinct top-level keys (`Johnson`, `Johnson2`) because a single
instrument cannot hold two identically-named sub-entries in this dict.

In [ ]:
local_filters = {
    '2MASS': {
        'J': {
            'J': '/mnt/home/glorenzon/analize_simba_cgm/2MASS_J.res',
        }
    },
    'Johnson': {
        'V': {
            'V': '/mnt/home/glorenzon/analize_simba_cgm/maiz-apellaniz_Johnson_V.res',
        }
    },
    # separate top-level key required: dict cannot hold two entries under 'Johnson'
    'Johnson2': {
        'U': {
            'U': '/mnt/home/glorenzon/analize_simba_cgm/maiz-apellaniz_Johnson_U.res',
        }
    },
}

### Stage 2 — extract photometric fluxes (run after RT completes)

Extracts fluxes for the **same** `(SNAPS, IDS)` in both runs. `extract_flux_batch` prints a
`requested / written / dropped` summary and writes `missing_sources.txt` listing any galaxy whose
`.rtout.sed` is absent or unreadable — so a count mismatch between the two runs is never silent.

In [ ]:
flux_files = {}
for key in RUNS:
    ms = makeseds[key]
    print(f"\n=== extracting fluxes: {key} (run_tag='{ms.run_tag}') ===")
    flux_file, xmean_file = ms.extract_flux_batch(
        SNAPS, IDS,
        ['HST', 'JWST', 'Spitzer', 'Herschel'],
        ['WFC3', 'NIRCam', 'IRAC', 'SPIRE'],
        filters=None, local_filters=local_filters, wave_unit='micron', findx=0,
    )
    flux_files[key] = flux_file

### Cross-check — both runs contain exactly the same sources

If the two runs disagree, the offending `(snap, gal)` pairs are listed here and in each run's
`missing_sources.txt`; chase those `.rtout.sed` files on the cluster (a 1-photon dust run can
crash or truncate output for some galaxies).

In [ ]:
present = {}
for key, ff in flux_files.items():
    t = Table.read(ff)
    present[key] = set(zip(np.asarray(t['snap'], int), np.asarray(t['gal_id_at_snap'], int)))

on, off = present['dust_on'], present['dust_off']
print(f"dust_on={len(on)}  dust_off={len(off)}  common={len(on & off)}")
print("only in dust_on :", sorted(on - off))
print("only in dust_off:", sorted(off - on))
print("OK — both runs contain exactly the same sources." if on == off
      else "MISMATCH — see the lists above and each run's missing_sources.txt")

### Photometric SEDs — dust_on vs dust_off

Per-galaxy multi-band fluxes vs effective filter wavelength, one panel per run (shared axes).

In [ ]:
meta_cols = {'gal_id_at_snap', 'snap', 'redshift'}

fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharex=True, sharey=True)
for ax, key in zip(axes, ['dust_on', 'dust_off']):
    flux_tab  = Table.read(flux_files[key])
    xmean_tab = Table.read(os.path.join(os.path.dirname(flux_files[key]), 'all_xmean.fits'))

    filter_cols = [c for c in flux_tab.colnames if c not in meta_cols]
    xmean_by_filter = {
        f: np.nanmedian(xmean_tab['xmean'][xmean_tab['filter'] == f]) for f in filter_cols
    }
    cols_sorted = sorted(filter_cols, key=lambda f: xmean_by_filter[f])
    wav_sorted  = np.array([xmean_by_filter[f] for f in cols_sorted])

    for row in flux_tab:
        fluxes = np.array([row[f] for f in cols_sorted], dtype=float)
        ax.plot(wav_sorted, fluxes, marker='o', ms=3, lw=0.8, alpha=0.7)

    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel(r'$\lambda_\mathrm{eff}\;[\mu\mathrm{m}]$')
    ax.set_title(f"{key}  (N={len(flux_tab)})")
axes[0].set_ylabel('Flux (mJy)')
fig.suptitle('Measured SED fluxes per galaxy — dust on vs off')
plt.tight_layout()
plt.show()